# iEEG/SEEG 色彩认知分析管道 (Color Cognition Pipeline) 框架引导

本 Notebook 作为整个色彩认知数据分析管道的**全局路线图与健康度检测文件**，用以梳理、引导和检测全管道的处理逻辑与数据产出。

## 1. 管道全景流程图 (Pipeline Roadmap)

该数据分析管道包含预处理、敏感电极筛选、记忆效应分析、感知-认知跨域泛化解码、一致性冲突检测、特异网络深度挖掘以及多频段窄带特征分析等多个主要模块：

```mermaid
graph TD
    A[Step 0: 数据准备与复制] --> B[Step 1: 颜色选择性电极筛选];
    B --> C[Step 1_2: 颜色选择性指数 CSI 量化];
    B --> D[Step 2_1: 灰色水果记忆颜色显著性分析];
    D --> E[Step 2_2/2_3: 颜色记忆解码 - SVM & GLMM];
    C & E --> F[Step 3: 真实与记忆色彩跨任务泛化解码 - Time-Time Generalization];
    E --> G[Step 4/6: 物体-颜色一致性违背解码 & 颞极 ERP 偏离];
    E --> H[Step 5: 电极网络 K-Means 聚类解码];
    B --> I[Step 7/8: Color_with_sti / Color_patch 特异网络深度挖掘与玻璃脑映射];
    B --> J[Step 9: D轴 & G轴电极多频段/高精度窄子带 (70-200Hz) 特征分析];
```

## 2. 全局配置与公共函数复用关系

为了消除冗余并降低维护成本，全管道公共的路径、常量以及统计/解剖学计算方法均已抽象至 [utils](file:///home/lirui/liulab_project/ieeg/Project_colorieeg_2026/color_cognition_pipeline/analyse_0617/code/utils) 子包中：

1. **路径与 Trigger 配置**：[utils/config.py](file:///home/lirui/liulab_project/ieeg/Project_colorieeg_2026/color_cognition_pipeline/analyse_0617/code/utils/config.py)  
2. **脑区匹配与物理电极**：[utils/anatomy.py](file:///home/lirui/liulab_project/ieeg/Project_colorieeg_2026/color_cognition_pipeline/analyse_0617/code/utils/anatomy.py)  
3. **均值计算与统计检验**：[utils/stats.py](file:///home/lirui/liulab_project/ieeg/Project_colorieeg_2026/color_cognition_pipeline/analyse_0617/code/utils/stats.py)  
4. **统一数据加载与平滑**：[utils/data_loader.py](file:///home/lirui/liulab_project/ieeg/Project_colorieeg_2026/color_cognition_pipeline/analyse_0617/code/utils/data_loader.py)

## 3. 全管道笔记本 (Notebook) 详细清单

| 步骤类别 | 笔记本文件链接 | 主要功能 | 核心输入 | 核心输出 |
| :--- | :--- | :--- | :--- | :--- |
| **数据准备** | [step0_data_check_copy.ipynb](file:///home/lirui/liulab_project/ieeg/Project_colorieeg_2026/color_cognition_pipeline/analyse_0617/code/step0_data_check_copy.ipynb) | 创建 feature 目录，复制 ERP 与 HG 信号 | 原始 processed_data/ | 工作区 feature/ 被试数据 |
| | [step0_1_preprocess_visualize.ipynb](file:///home/lirui/liulab_project/ieeg/Project_colorieeg_2026/color_cognition_pipeline/analyse_0617/code/step0_1_preprocess_visualize.ipynb) | 对比未滤波、滤波、三极重参考以及 HG 融合预处理效果 | `erp1.set` 原始信号 | `step0_1_G11_preprocessing_comparison.png` |
| **敏感筛选** | [step1_select_channel.ipynb](file:///home/lirui/liulab_project/ieeg/Project_colorieeg_2026/color_cognition_pipeline/analyse_0617/code/step1_select_channel.ipynb) | 脑定位关联大类，通过 Task 1 颜色 vs 灰色对比筛选敏感电极 | Task 1 ERP/HG | `doc/select_channel_summary.xlsx` |
| | [step1_1_select_channel_extended.ipynb](file:///home/lirui/liulab_project/ieeg/Project_colorieeg_2026/color_cognition_pipeline/analyse_0617/code/step1_1_select_channel_extended.ipynb) | 扩展电极筛选准则，输出详细统计表及单电极 ERP 波形 | Task 1 ERP/HG | 详细电极表及 ERP 波形图 |
| | [step1_2_color_selectivity.ipynb](file:///home/lirui/liulab_project/ieeg/Project_colorieeg_2026/color_cognition_pipeline/analyse_0617/code/step1_2_color_selectivity.ipynb) | 计算敏感电极在 Task 3 中的色彩选择性指数 (CSI) | Task 3 数据 | `doc/select_channel_color_selectivity_erp/hg.csv` |
| | [step1_2_color_selectivity_correlation.ipynb](file:///home/lirui/liulab_project/ieeg/Project_colorieeg_2026/color_cognition_pipeline/analyse_0617/code/step1_2_color_selectivity_correlation.ipynb) | 统计 CSI 与被试脑区 MNI Y 轴的相关性 | CSI 结果文件 | `color_selectivity_mni_y_correlation.png` |
| | [step1_2_color_selectivity_paper_formula.ipynb](file:///home/lirui/liulab_project/ieeg/Project_colorieeg_2026/color_cognition_pipeline/analyse_0617/code/step1_2_color_selectivity_paper_formula.ipynb) | 论文标准 $(R_{color}-R_{BW})/(R_{color}+R_{BW})$ 计算 CSI 并分析 MNI Y 相关性 | Task 3 数据 | CSI 分布图及回归图 |
| **记忆效应** | [step2_1_memory_color_significance.ipynb](file:///home/lirui/liulab_project/ieeg/Project_colorieeg_2026/color_cognition_pipeline/analyse_0617/code/step2_1_memory_color_significance.ipynb) | 评估灰色红/绿水果下敏感电极反应差异显著性 | Task 2 灰色水果 | 单电极显著波形图及 ERP/HG 显著统计表 |
| | [step2_2_memory_color_decoding_glmm.ipynb](file:///home/lirui/liulab_project/ieeg/Project_colorieeg_2026/color_cognition_pipeline/analyse_0617/code/step2_2_memory_color_decoding_glmm.ipynb) | 多通道 SVM 跨水果灰色解码与混合模型显著窗口估计 | Task 2 灰色水果 | 记忆颜色群体解码准确率时程图 |
| | [step2_3_single_electrode_decoding_correlation.ipynb](file:///home/lirui/liulab_project/ieeg/Project_colorieeg_2026/color_cognition_pipeline/analyse_0617/code/step2_3_single_electrode_decoding_correlation.ipynb) | 对各敏感电极单独做灰色红绿解码，探查与 MNI Y 的相关性 | Task 2 灰色水果 | 解码准确率时空格局热图 |
| **感知-认知** | [step3_1_color_block_decoding.ipynb](file:///home/lirui/liulab_project/ieeg/Project_colorieeg_2026/color_cognition_pipeline/analyse_0617/code/step3_1_color_block_decoding.ipynb) | 对 Task 3 纯色块进行滑动 SVM 交叉验证解码并做 GLMM 检验 | Task 3 纯色块 | 真实颜色感知解码曲线 |
| | [step3_2_cross_decoding_generalization.ipynb](file:///home/lirui/liulab_project/ieeg/Project_colorieeg_2026/color_cognition_pipeline/analyse_0617/code/step3_2_cross_decoding_generalization.ipynb) | 真实色彩感知 (Task 3) 和记忆色彩 (Task 2) 之间的跨任务时间泛化解码 | Task 2 与 Task 3 | 时间-时间解码泛化矩阵热图 |
| | [step3_2_cross_decoding_generalization_union.ipynb](file:///home/lirui/liulab_project/ieeg/Project_colorieeg_2026/color_cognition_pipeline/analyse_0617/code/step3_2_cross_decoding_generalization_union.ipynb) | 使用敏感电极并集 (Union) 重做跨任务时间泛化解码 | Task 2 与 Task 3 | 泛化矩阵热图 (`_union`) |
| | [step3_3_draw_B5_color_block_erp.ipynb](file:///home/lirui/liulab_project/ieeg/Project_colorieeg_2026/color_cognition_pipeline/analyse_0617/code/step3_3_draw_B5_color_block_erp.ipynb) | 绘制代表电极 `test001-B5` 在 Task 3 纯色刺激下的 ERP | Task 3 ERP | B5 纯色 ERP 与均值对照图 |
| | [step3_3_single_electrode_generalization.ipynb](file:///home/lirui/liulab_project/ieeg/Project_colorieeg_2026/color_cognition_pipeline/analyse_0617/code/step3_3_single_electrode_generalization.ipynb) | 各主要通道单独计算跨感知-记忆的时间泛化矩阵 | Task 2 与 Task 3 | 各电极专属的时间泛化矩阵图 |
| **一致冲突** | [step4_real_fake_color_decoding.ipynb](file:///home/lirui/liulab_project/ieeg/Project_colorieeg_2026/color_cognition_pipeline/analyse_0617/code/step4_real_fake_color_decoding.ipynb) | 机器学习解码真彩色 (红草莓) 与假彩色 (绿草莓)，按 ROI 统计 | Task 2 8个触发器 | 脑区真假冲突解码曲线及数据表 |
| | [step5_memory_color_clusters_decoding.ipynb](file:///home/lirui/liulab_project/ieeg/Project_colorieeg_2026/color_cognition_pipeline/analyse_0617/code/step5_memory_color_clusters_decoding.py) | 基于 MNI_Y 坐标进行电极 K-Means 聚类，比较前部/后部脑区解码 | Task 2 数据 | 聚类热图及不同 Cluster 解码时程图 |
| | [step6_temporal_pole_true_fake_erp_difference.ipynb](file:///home/lirui/liulab_project/ieeg/Project_colorieeg_2026/color_cognition_pipeline/analyse_0617/code/step6_temporal_pole_true_fake_erp_difference.ipynb) | 提取高级颞极（Temporal Pole）敏感电极，分析 True/Fake 间 ERP 差异 | Task 2 颞极数据 | 颞极电极 ERP 差值与显著段标记 |
| **特异分析** | [step7_color_with_sti_electrode_analyses.ipynb](file:///home/lirui/liulab_project/ieeg/Project_colorieeg_2026/color_cognition_pipeline/analyse_0617/code/step7_color_with_sti_electrode_analyses.ipynb) | 针对定位为 `color_with_sti` 的电极群集进行 CSI/显著性/解码/泛化全套计算 | 定位表中特异通道 | 特异性网络下全套图表 |
| | [step8_2_whole_brain_erp_strategy_table_and_glassbrain.ipynb](file:///home/lirui/liulab_project/ieeg/Project_colorieeg_2026/color_cognition_pipeline/analyse_0617/code/step8_2_whole_brain_erp_strategy_table_and_glassbrain.ipynb) | 排除特异电极后对全脑电极做 4 策略筛选，绘制 3D Nilearn 玻璃脑 | Task 1 数据，坐标表 | `whole_brain_erp_glass_brain.png` |
| | [step8_cws_brain_and_memory_erp_latency.ipynb](file:///home/lirui/liulab_project/ieeg/Project_colorieeg_2026/color_cognition_pipeline/analyse_0617/code/step8_cws_brain_and_memory_erp_latency.ipynb) | `color_with_sti` 玻璃脑渲染；计算记忆显著电极的潜伏期 (ESTP) 并探查与 MNI Y 关联 | 记忆电极与坐标表 | 2D 玻璃脑、ESTP vs MNI_Y 相关图 |
| **特征挖掘** | [step9_electrode_d_features_and_plots.ipynb](file:///home/lirui/liulab_project/ieeg/Project_colorieeg_2026/color_cognition_pipeline/analyse_0617/code/step9_electrode_d_features_and_plots.ipynb) | 对 D 轴电极行拉普拉斯/双极重参考，提取 ERP/Gamma/HG/PSD 特征 | SEEG 数据 | D 轴电极多指标分类对比图 |
| | [step9_2_electrode_g_features_and_plots.ipynb](file:///home/lirui/liulab_project/ieeg/Project_colorieeg_2026/color_cognition_pipeline/analyse_0617/code/step9_2_electrode_g_features_and_plots.ipynb) | 对 G 轴电极行局部重参考，计算并绘制多频段成分 | SEEG 数据 | G 轴电极多成分时程及功率图 |
| | [step9_3_subband_70_200hz_plots.ipynb](file:///home/lirui/liulab_project/ieeg/Project_colorieeg_2026/color_cognition_pipeline/analyse_0617/code/step9_3_subband_70_200hz_plots.ipynb) | 在 D/G 电极上提取极高频 (70-200Hz) 精细窄频带包络特征并做对比 | SEEG 数据 | 5x2 窄带融合时程与均值柱状图 |

## 4. 全管道健康度自检

你可以直接执行下方的代码单元格，它会自动扫描 `doc/` 和 `result/` 目录，确认各个分析步骤的关键数据输出（Excel/CSV）与关键图像是否齐备，并生成健康度报告表格。

In [ ]:
import os
import pandas as pd
from utils.config import DOC_DIR, RESULT_DIR

# 定义需要检查的核心产出文件
target_files = {
    "Step 1: 筛选通道表": os.path.join(DOC_DIR, "select_channel_summary.xlsx"),
    "Step 1_2: 颜色选择性 ERP 强偏好": os.path.join(DOC_DIR, "select_channel_color_selectivity_erp.xlsx"),
    "Step 1_2: 颜色选择性 HG 强偏好": os.path.join(DOC_DIR, "select_channel_color_selectivity_hg.xlsx"),
    "Step 1_2: 论文版相关性数据": os.path.join(DOC_DIR, "color_selectivity_correlation_summary_paper.xlsx"),
    "Step 2_1: ERP 记忆显著性表": os.path.join(DOC_DIR, "select_channel_memory_significance_erp.xlsx"),
    "Step 2_3: ERP 记忆单电极解码 ESTP": os.path.join(DOC_DIR, "select_channel_memory_decoding_estp_erp.xlsx"),
    "Step 3_2: 跨解码泛化 Excel": os.path.join(DOC_DIR, "cross_decoding_tg_strategy1.xlsx"),
    "Step 4: 颞极真假冲突解码数据": os.path.join(DOC_DIR, "real_fake_decoding_results_temporal_pole.xlsx"),
    "Step 5: 聚类 anterior 记忆解码": os.path.join(DOC_DIR, "decoding_data_erp_cluster_anterior_memory_color.xlsx"),
    "Step 5: 聚类 posterior 记忆解码": os.path.join(DOC_DIR, "decoding_data_erp_cluster_posterior_memory_color.xlsx"),
    "Step 6: 颞极真假 ERP 差异统计": os.path.join(DOC_DIR, "temporal_pole_true_fake_erp_stats.xlsx"),
    "Step 8_2: 全脑 ERP 策略表格": os.path.join(DOC_DIR, "whole_brain_erp_strategy_summary.xlsx"),
    "Step 8_cws: 记忆 ERP ESTP 潜伏期表": os.path.join(DOC_DIR, "memory_color_erp_signal_diff_estp.xlsx"),
}

print("=================== Pipeline Output Health Checker ===================")
res_list = []
for key, path in target_files.items():
    exists = os.path.exists(path)
    size_kb = os.path.getsize(path) / 1024 if exists else 0.0
    status = "✅ OK" if exists else "❌ Missing"
    res_list.append({
        "分析产出指标": key,
        "检查状态": status,
        "大小 (KB)": f"{size_kb:.2f} KB" if exists else "-",
        "物理绝对路径": path
    })

df_res = pd.DataFrame(res_list)
# 设置 pandas 显示格式以便于预览
pd.set_option('display.max_colwidth', None)
display(df_res)
print("======================================================================")